# 01 - Dataset Audit

#### Setup configuration

In [1]:
from IPython.display import display
from pathlib import Path

import numpy as np
import pandas as pd

DATASET_DIR = Path("../dataset")

FEATURES_PATH = DATASET_DIR / "elliptic_txs_features.csv"
CLASSES_PATH = DATASET_DIR / "elliptic_txs_classes.csv"
EDGES_PATH = DATASET_DIR / "elliptic_txs_edgelist.csv"

In [2]:
feature_columns = [
    "tx_id",
    "time_step",
    *[f"feature_{i}" for i in range(1, 166)],
]

features_df = pd.read_csv(FEATURES_PATH, header=None, names=feature_columns)

classes_df = pd.read_csv(CLASSES_PATH).rename(
    columns={"txId": "tx_id", "class": "label"}
)

edges_df = pd.read_csv(EDGES_PATH).rename(
    columns={"txId1": "source_tx_id", "txId2": "target_tx_id"}
)

#### Core node features

In [3]:
print("FEATURES")
display(features_df.head())
print(features_df.shape)

print("\nCLASSES")
display(classes_df.head())
print(classes_df.shape)

print("\nEDGES")
display(edges_df.head())
print(edges_df.shape)

FEATURES


,tx_id,time_step,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


(203769, 167)

CLASSES


,tx_id,label
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


(203769, 2)

EDGES


,source_tx_id,target_tx_id
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


(234355, 2)


In [4]:
print(features_df.dtypes.head(10))
print()
print(classes_df.dtypes)
print()
print(edges_df.dtypes)

tx_id          int64
time_step      int64
feature_1    float64
feature_2    float64
feature_3    float64
feature_4    float64
feature_5    float64
feature_6    float64
feature_7    float64
feature_8    float64
dtype: object

tx_id    int64
label      str
dtype: object

source_tx_id    int64
target_tx_id    int64
dtype: object


#### Dataset validation

In [5]:
duplicates = {
    "features tx_id": features_df["tx_id"].duplicated().sum(),
    "classes tx_id": classes_df["tx_id"].duplicated().sum(),
    "edges": edges_df.duplicated().sum(),
}

pd.Series(duplicates, name="duplicate_count")

features tx_id    0
classes tx_id     0
edges             0
Name: duplicate_count, dtype: int64

In [6]:
missing_values = {
    "features": int(features_df.isna().sum().sum()),
    "classes": int(classes_df.isna().sum().sum()),
    "edges": int(edges_df.isna().sum().sum()),
}

numeric_features = features_df.drop(columns="tx_id")
infinite_values = int(np.isinf(numeric_features.to_numpy()).sum())

print("Missing values:")
display(pd.Series(missing_values).to_frame().T)

print("Infinite feature values:", infinite_values)

Missing values:


,features,classes,edges
0,0,0,0


Infinite feature values: 0


#### Timesteps

In [7]:
time_steps = sorted(features_df["time_step"].unique())

print("Number of time steps:", len(time_steps))
print("Minimum time step:", min(time_steps))
print("Maximum time step:", max(time_steps))
print("Time steps:", time_steps)

Number of time steps: 49
Minimum time step: 1
Maximum time step: 49
Time steps: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49)]


#### Classes

In [8]:
label_counts = classes_df["label"].astype(str).value_counts()

display(label_counts.rename("count").to_frame())

,count
label,
unknown,157205
2,42019
1,4545


#### Transactions

In [9]:
transactions_df = features_df.merge(
    classes_df,
    on="tx_id",
    how="left",
    validate="one_to_one",
)

label_mapping = {
    "1": "illicit",
    "2": "licit",
    "unknown": "unknown",
}

transactions_df = transactions_df.copy()

transactions_df["label_name"] = transactions_df["label"].astype(str).map(label_mapping)

display(transactions_df.head())
print("Transactions after merge:", transactions_df.shape)
print("Labels missing after merge:", transactions_df["label"].isna().sum())

,tx_id,time_step,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,label,label_name
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,unknown,unknown
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,unknown,unknown
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792,unknown,unknown
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792,2,licit
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117,unknown,unknown


Transactions after merge: (203769, 169)
Labels missing after merge: 0


#### Overall statistics

In [10]:
feature_nodes = set(features_df["tx_id"])

edge_nodes = set(edges_df["source_tx_id"]) | set(edges_df["target_tx_id"])

missing_edge_nodes = edge_nodes - feature_nodes
isolated_nodes = feature_nodes - edge_nodes

print("Nodes in features:", len(feature_nodes))
print("Nodes occurring in edges:", len(edge_nodes))
print("Edge nodes missing from features:", len(missing_edge_nodes))
print("Isolated nodes:", len(isolated_nodes))

Nodes in features: 203769
Nodes occurring in edges: 203769
Edge nodes missing from features: 0
Isolated nodes: 0


In [11]:
node_time_steps = features_df.set_index("tx_id")["time_step"]

edges_with_time = edges_df.assign(
    source_time_step=edges_df["source_tx_id"].map(node_time_steps),
    target_time_step=edges_df["target_tx_id"].map(node_time_steps),
)

cross_time_edges = edges_with_time[
    edges_with_time["source_time_step"] != edges_with_time["target_time_step"]
]

print("Edges between different time steps:", len(cross_time_edges))

Edges between different time steps: 0


In [12]:
audit_summary = pd.Series(
    {
        "transactions": len(features_df),
        "edges": len(edges_df),
        "columns_in_features_file": features_df.shape[1],
        "model_features_including_time_step": features_df.shape[1] - 1,
        "time_steps": features_df["time_step"].nunique(),
        "illicit_transactions": int(label_counts.get("1", 0)),
        "licit_transactions": int(label_counts.get("2", 0)),
        "unknown_transactions": int(label_counts.get("unknown", 0)),
        "duplicated_feature_tx_ids": int(features_df["tx_id"].duplicated().sum()),
        "missing_feature_values": int(features_df.isna().sum().sum()),
        "infinite_feature_values": infinite_values,
        "edge_nodes_missing_from_features": len(missing_edge_nodes),
        "isolated_nodes": len(isolated_nodes),
        "cross_time_edges": len(cross_time_edges),
    },
    name="value",
)

display(audit_summary.to_frame())

,value
transactions,203769
edges,234355
columns_in_features_file,167
model_features_including_time_step,166
time_steps,49
illicit_transactions,4545
licit_transactions,42019
unknown_transactions,157205
duplicated_feature_tx_ids,0
missing_feature_values,0


In [13]:
assert features_df.shape == (203_769, 167)
assert classes_df.shape == (203_769, 2)
assert edges_df.shape == (234_355, 2)

assert features_df["tx_id"].is_unique
assert classes_df["tx_id"].is_unique

assert len(time_steps) == 49
assert time_steps == list(range(1, 50))

assert label_counts.to_dict() == {
    "unknown": 157_205,
    "2": 42_019,
    "1": 4_545,
}

assert not missing_edge_nodes
assert not isolated_nodes
assert cross_time_edges.empty

print("All audit checks passed.")

All audit checks passed.
